# DSCI 100 Final Project Report

## (1) Introduction:

A research group in Computer Science at UBC, led by Dr. Frank Wood, is gathering data on video game behavior through a Minecraft server. Their players.csv dataset contains 196 unique observations with 7 variables that capture demographic information and player skill levels as follows:

- **experience**: Describes the player's level of gaming experience. (*fct, loaded as chr*)
- **subscribe**: Indicates whether the player is subscribed to the game's associated newsletter. (*lgl*)
- **hashedEmail**: The player's email encrypted as a code. (*chr*)
- **played_hours**: The number of hours the player has played on the Minecraft server. (*dbl*)
- **name**: The player's first name. (*chr*)
- **gender**: The player's gender. (*fct, loaded as chr*) 
- **Age**: The player's age in years. (*int, loaded as double*)

Our project investigates which player characteristics and behaviors predict newsletter subscription and how these predictors differ among various player types. Specifically, we ask: can a player's age (`Age`) and number of hours player on the server (`played_hours`) predict whether they are subscribed to the newsletter, according to the `players` dataset? This question supports Dr. Frank Wood's research by helping to target recruitment efforts toward individuals likely to subscribe. Their willingness to subscribe suggests they may be more engaged and responsive, making them better subjects for ongoing studies. Thus, identifying the predictors that reveal which types of players have an increased tendency to subscribe will enable better recruitment. Unfortunately, it is unclear if data submission was mandatory for all players; if only a subset contributed, the dataset may not represent the entire playerbase, limiting its usefulness for building an effective classifier.

## (2) Methods:

The `players` data from the players.csv wi

As mentioned above, the data that will help us address our question is the players.csv file. KNN classification will be used to predict a player’s subscription status based on age and hours played. Age and number of hours played will both be scaled so that they both contribute equally to the KNN model. Subscription status will also be changed to a factor type of data, and NAs will be dropped to allow KNN classification to work. The model will be tested with a test set of data. This model assumes that players input their real age, and a potential limitation is that it doesn’t account for gender or experience, which were also given in the players.csv dataset. To process the data, we will split the data into test (30%) and training sets (70%). The training set will be used to perform 5 fold cross-validation to determine the best value of k. Using this value of k, we will test the KNN classification model using the test set of data to determine how accurate the model is at predicting a player’s subscription status based on age and hours played. We include accuracy, recall, and precision metrics of the KNN classifier to help show the effectiveness of the model.

## (3) Code and Results:

In [ ]:
library(tidyverse)
library(tidymodels)
library(cowplot)
library(repr)

set.seed(4)
options(repr.plot.width = 14)

players_url <- "https://raw.githubusercontent.com/oo74/DSCI-100-Project/d932a95bab3bbe9a443dcba02939882b0735483f/data/players.csv"
players <- read_csv(players_url) |>
    mutate(subscribe = fct_recode(as_factor(subscribe), Yes = "TRUE", No = "FALSE"))


nrow(players)

players |>
    map_df(n_distinct)

players |>
    group_by(subscribe) |>
    summarize(count = n())


players |>
    summarize(mean = mean(played_hours, na.rm = TRUE),
              SD = sd(played_hours, na.rm = TRUE),
              min = min(played_hours, na.rm = TRUE),
              max = max(played_hours, na.rm = TRUE),
              median = median(played_hours, na.rm = TRUE))

players |>
    summarize(mean = mean(Age, na.rm = TRUE),
              SD = sd(Age, na.rm = TRUE),
              min = min(Age, na.rm = TRUE),
              max = max(Age, na.rm = TRUE),
              median = median(Age, na.rm = TRUE))

The `played_hours` variable spans a wide range—from 0 to 223.1 hours—with a standard deviation of 28.36 hours, indicating considerable variability; however, a mean of 5.85 hours and a median of 0.1 hours suggest that most players have very few recorded hours. However, it is unclear whether these low hours are a consequence of the players being new or a genuine lack of interest in gaming. This distinction is important for predictions based on `played_hours`, as this metric may not accurately capture a player's typical gaming behavior. Though `Age` ranges from 8 to 50 years, the mean of 20.5 years, the median of 19 years, and the modest standard deviation of 6.17 indicates that most players are relatively young.

| Variable Name | No. of Unique Values | Mean | Standard Deviation | Min | Max | Median |
| -------- | ------- | ------- | ------- | ------- | ------- | ------- |
| subscribe | 2 |
| played_hours | 43 | 5.845918 |28.35734 | 0 | 223.1 | 0.1 |
| Age | 31 | 20.52062 | 6.174667 | 8 | 50 | 19 |

The `subscribe` variable was converted from a logical to a factor type using `fct_recode()` for proper KNN classification labeling. The data includes 142 newsletter subscribers and 52 non-subscribers, indicating imbalanced classes for `subscribe`. 

In [ ]:
age_hours_plot <- players |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 1: Age, played hours, and actual subscription status of all players.", color = "Subscribed?") +
    theme(text = element_text(size = 16), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

age_plot <- players |>
    ggplot(aes(x = Age, fill = subscribe)) +
    geom_histogram(binwidth = 1) +
    labs(x = "Age (years)", y = "Count", title = "Fig. 2: Distribution of age of players and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))

hours_plot <- players |>
    ggplot(aes(x = played_hours, fill = subscribe)) +
    geom_histogram(binwidth = 2) +
    labs(x = "Played Hours", y = "Count", title = "Fig. 3: Distribution of players' played hours and their corresponding subscription status.") +
    theme(text = element_text(size = 10), legend.position = "bottom", legend.direction = "horizontal") +
    scale_fill_manual(values = c("red", "chartreuse3"))


age_hours_plot
plot_grid(age_plot, hours_plot, ncol = 2)

To reduce overplotting and enhance density visualization, datapoint opacity was set to 0.6, where higher opacity indicates more overlap. 

Fig. 1 shows no clear linear relationship between hours played and age. Most players are clustered around zero hours, with a few outliers exceeding 150 hours. As seen in Fig. 2, the majority of players are between 15 and 28 years old, with a significant concentration at age 17. Nearly all age groups 17 and older contain non-subscribed players, while those younger are all subscribed. Fig. 3 indicates that most non-subscribed players have low played hours. However, since the majority of all players report low hours and there is limited data for high-hour players, this observation may not imply a direct association between low played hours and the tendency to subscribe.

In [ ]:
players <- drop_na(players)
nrow(players)

k_vals <- tibble(neighbors = 1:25)

players_split <- initial_split(players, prop = 0.75, strata = subscribe)
players_train <- training(players_split)
players_test <- testing(players_split)

players_recipe <- recipe(subscribe ~ played_hours + Age, data = players_train) |>
    step_center(all_predictors()) |>
    step_scale(all_predictors()) 

players_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
    set_engine("kknn") |>
    set_mode("classification")

vfold_sets <- players_train |>
    vfold_cv(v = 5, strata = subscribe)

k_accuracies <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_spec) |>
    tune_grid(resamples = vfold_sets, grid = k_vals) |>
    collect_metrics() |>
    filter(.metric == "accuracy") |>
    mutate(accuracy = mean) |>
    select(neighbors, accuracy)
    
k_accuracies_plot <- k_accuracies |>
    ggplot(aes(x = neighbors, y = accuracy)) +
    geom_point() +
    geom_line() +
    labs(x = "Neighbors (K)", y = "Accuracy", title = "Fig. 4: Accuracies associated with various K values.") +
    theme(text = element_text(size = 12))

best_k <- k_accuracies |>
    slice_max(accuracy) |>
    slice_min(neighbors) |>
    pull(neighbors)

best_k
k_accuracies_plot

The 2 observations with `NA` were removed to avoid interference with classification. This small loss is unlikely to have a significant impact on the data.

Before modeling, 75% of the data was split into a training set and 25% was split into a testing set. This ensures that the model's performance is evaluated on unseen data rather than on the same examples it was trained on, which can lead to inaccurately high performance metrics and poor classification of new observations if the model simply memorizes the training data instead of learning generalizable patterns. To determine the best K value for the K-nearest neighbors classifier, 5-fold cross-validation was performed on the training set for K values ranging from 1 to 25, as experimentation with a smaller range of K's showed that the accuracy kept increasing as K increased and did not plateauing until K was around 20. The mean accuracies were calculated for each K, and the best K was selected based on the highest accuracy. Any ties among the highest accuracy K values were resolved by selecting the smallest K to improve computational efficiency.

As seen in Fig. 4, accuracy increases as K increases, plateauing at around 20. Accuracy is first maximized when K is 17.

In [ ]:
players_best_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = best_k) |>
    set_engine("kknn") |>
    set_mode("classification")

players_fit <- workflow() |>
    add_recipe(players_recipe) |>
    add_model(players_best_spec) |>
    fit(data = players_train)

players_predicted <- players_fit |>
    predict(players_test) |>
    bind_cols(players_test) 

players_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = subscribe)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 5: Age, played hours, and actual subscription status of players in test set.", color = "Subscribed?") +
    ylim(0, 2.5) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = c("red", "chartreuse3"))

players_predicted_plot <- players_predicted |>
    ggplot(aes(x = Age, y = played_hours, color = .pred_class)) +
    geom_point(alpha = 0.6) +
    labs(x = "Age (years)", y = "Played Hours", title = "Fig. 6: Age, played hours, and predicted subscription status of players in test set", color = "Predicted to subscribe?") +
    ylim(0, 2.5) +
    theme(text = element_text(size = 11), legend.position = "bottom", legend.direction = "horizontal") +
    scale_color_manual(values = "chartreuse3")

plot_grid(players_plot, players_predicted_plot, ncol = 2)

With the optimal K determined, a new model was built using that K to predict outcomes on the test set. Since initial visualizations revealed that most observations reported under 2.5 player hours, the y-axis was capped at 2.5 to spread out the majority of the datapoints and enhance clarity.

In [ ]:
players_accuracy <- players_predicted |>
    metrics(truth = subscribe, estimate = .pred_class) |>
    filter(.metric == "accuracy")

players_precision <- players_predicted |>
    precision(truth = subscribe, estimate = .pred_class, event_level = "second")

players_recall <- players_predicted |>
    recall(truth = subscribe, estimate = .pred_class, event_level = "second")

players_conf_mat <- players_predicted |>
    conf_mat(truth = subscribe, estimate = .pred_class)

players_conf_mat
bind_rows(players_accuracy, players_precision, players_recall) |>
    select(-.estimator)

Model performance metrics were calculated with "Yes" as the positive class for the variable `subscribe`. The model achieved an accuracy of 0.7346939, a perfect recall of 1, and a precision of 0.7346939. The confusion matrix and Fig. 6 indicate that the model predicted every observation as subscribed ("Yes").

## (4) Discussion:
- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?

The accuracy of played might seem solid initially, but an accuracy of 0.73 indicates that nearly 3 in every 10 individuals are misclassified, suggesting age and played hours are likely poor predictors of subscription status. The model's perfect recall is a result of predicting "Yes" for every observation, which also explains why the precision matches the accuracy, as the denominators (total number of predictions equals the number of positive predictions) and the numerators (number of true predictions equals number of true positive predictions) are the same. These misleading performance metrics likely stem from class imbalance. With relatively few negatives ("No"), the model produces fewer false positives even when it predicts "Yes" for every observation, inflating accuracy and precision. Consequently, it remains unclear whether age and played hours are inherently poor predictors or if the imbalance is obscuring their true potential. Upsampling the "No" class during pre-processing could help address this issue. Initially, we expected that a higher number of hours played, and a lower age may be indicative of being subscribed to the newsletter. However, as mentioned above, the findings from this data analysis was largely inconclusive. These inconclusive findings may prevent the developers from being able to target a demographic to advertise to in order to increase the levels of subscription. Overall, this project raises the question of how well age and number of hours played may be at predicting a players subscription status if the number of non-subscribed people were upsampled. If upsampling shows that age and number of hours played are good indicators of a players subscription status, then this could help the developers of the game with their marketing efforts. However, if analysis of data with upsampling still indicates age and number of hours played as poor indicators of subscription status, then this could raise further questions about what other variables may be indicative of subscription status then. 